# Chapter 4: aggregate thesis results

This notebook aggregates **saved observations**, without importing or executing the
optimization/training package. Run cells in order. Dependencies: NumPy and pandas
(plus Jupyter/ipykernel for interactive use).

- **Table 4.3:** recompute from the available historical applicant CSVs and check
  against both their saved summary and the thesis's rounded values.
- **Tables 4.1–4.2:** aggregate recovered quadratic CSVs when supplied. No historical
  raw CSVs were present in the original folder. Printed thesis values below are
  explicitly labeled references and are never substituted for missing observations.
- Historical algorithm defects remain unfixed. Read [findings](../docs/findings.md)
  and [aggregation policy](../docs/reproduction.md). Stored numbers are not proof of
  statistical guarantees or exact retraining/solver reproducibility.


In [1]:
from pathlib import Path
import hashlib
import json
import re
import numpy as np
import pandas as pd

def find_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "conformal_cl").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter from the repository or set ROOT explicitly.")

ROOT = find_root(Path.cwd().resolve())
ENROLLMENT_DIR = ROOT / "results" / "enrollment"
QUADRATIC_FILES = sorted((ROOT / "results" / "quadratic").glob("*.csv"))
QUADRATIC_FILTERS = {"alpha": [0.1], "P": [10], "dim": [1, 2, 3], "noise_std": [0.1, 0.15]}
FEASIBILITY_DENOMINATOR = "all_true_feasible"  # or "returned_solutions"
TIMING_POPULATION = "all_true_feasible"  # or "returned_solutions", "all_instances"
EXPORT = True
EXPORT_DIR = ROOT / "results" / "aggregated"
INPUT_HASHES = {}

def read_result_csv(path):
    path = Path(path)
    key = str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)
    INPUT_HASHES[key] = hashlib.sha256(path.read_bytes()).hexdigest()
    return pd.read_csv(path, encoding="utf-8-sig")

print("Enrollment inputs:", ENROLLMENT_DIR.relative_to(ROOT))
print("Quadratic files:", len(QUADRATIC_FILES))
print("Feasibility denominator:", FEASIBILITY_DENOMINATOR)
print("Timing population:", TIMING_POPULATION)


Enrollment inputs: results/enrollment
Quadratic files: 0
Feasibility denominator: all_true_feasible
Timing population: all_true_feasible


## Table 4.3 — enrollment case study (§4.6.3)

Merit is stored in thousands of dollars. Validate StudentID, applicant features,
theta=.17, statuses, and merit values. Means use optimal rows only; counts make
the denominator explicit. In the historical files all 6,000 rows per method are optimal.


In [2]:
ENROLLMENT_METHODS = {"basic": "Basic Model", "quantile": "Quantile Model", "rcc": "RCC Model"}
THESIS_4_3 = {"Basic Model": 1021.17, "Quantile Model": 1054.94, "RCC Model": 1238.31}

def aggregate_enrollment(directory):
    records, by_method = [], {}
    reference_features = None
    for suffix, label in ENROLLMENT_METHODS.items():
        frame = read_result_csv(directory / f"enrollment_result_{suffix}.csv")
        required = {"StudentID", "SAT", "GPA", "OptimalMerit", "Status", "Theta"}
        if not required.issubset(frame.columns):
            raise ValueError(f"{suffix}: missing columns {required - set(frame.columns)}")
        if frame.empty or frame["StudentID"].isna().any() or frame["StudentID"].duplicated().any():
            raise ValueError(f"{suffix}: empty data, missing IDs, or duplicate IDs")
        numeric = frame[["SAT", "GPA", "Theta"]].apply(pd.to_numeric, errors="raise")
        if not np.isfinite(numeric.to_numpy()).all() or not np.allclose(numeric["Theta"], 0.17, rtol=0, atol=1e-12):
            raise ValueError(f"{suffix}: invalid features or mixed/non-thesis theta")
        if not frame["Status"].isin(["Optimal", "Infeasible"]).all():
            raise ValueError(f"{suffix}: unrecognized status")
        features = frame.set_index("StudentID")[["SAT", "GPA"]].sort_index()
        if reference_features is None:
            reference_features = features
        elif not features.equals(reference_features):
            raise ValueError(f"{suffix}: applicant cohort/features do not match")
        merit = pd.to_numeric(frame["OptimalMerit"], errors="raise")
        optimal = frame["Status"].eq("Optimal")
        if not np.isfinite(merit[optimal]).all() or not merit[optimal].between(-1e-6, 25 + 1e-6).all():
            raise ValueError(f"{suffix}: invalid optimal merit")
        if merit[~optimal].notna().any():
            raise ValueError(f"{suffix}: nonoptimal row contains a merit solution")
        records.append({"Method": label, "Total_Applicants": len(frame),
                        "Optimal_Solutions": int(optimal.sum()), "Infeasible_Solutions": int((~optimal).sum()),
                        "Average_Merit_Dollars": merit[optimal].mean() * 1000, "Theta": 0.17})
        by_method[suffix] = frame.set_index("StudentID")
    return pd.DataFrame(records), by_method

table_4_3, enrollment_rows = aggregate_enrollment(ENROLLMENT_DIR)
table_4_3["Thesis_USD"] = table_4_3["Method"].map(THESIS_4_3)
table_4_3["Matches_Thesis_2dp"] = table_4_3["Average_Merit_Dollars"].round(2).eq(table_4_3["Thesis_USD"])
display(table_4_3)

saved_summary_path = ENROLLMENT_DIR / "enrollment_result_comparison.csv"
if saved_summary_path.exists():
    saved_summary = read_result_csv(saved_summary_path)
    if saved_summary["Method"].duplicated().any() or set(saved_summary["Method"]) != set(table_4_3["Method"]):
        raise ValueError("Saved comparison has duplicate/mismatched method labels")
    checked = table_4_3.merge(saved_summary, on="Method", validate="one_to_one", suffixes=("_computed", "_saved"))
    for name in ["Total_Applicants", "Optimal_Solutions", "Infeasible_Solutions", "Theta", "Average_Merit_Dollars"]:
        if not np.allclose(checked[f"{name}_computed"], checked[f"{name}_saved"], rtol=0, atol=1e-8, equal_nan=True):
            raise ValueError(f"Saved comparison does not match applicant aggregation: {name}")
    print("All recomputed values match the saved comparison.")
else:
    print("No saved comparison CSV: applicant-level computation remains available.")
print("All three averages match rounded thesis values:", bool(table_4_3["Matches_Thesis_2dp"].all()))
print("Historical status labels do not distinguish mathematical infeasibility from solver failure (ISSUE-09).")


Method,Total_Applicants,Optimal_Solutions,Infeasible_Solutions,Average_Merit_Dollars,Theta,Thesis_USD,Matches_Thesis_2dp
Basic Model,6000,6000,0,1021.167124,0.17,1021.17,True
Quantile Model,6000,6000,0,1054.939617,0.17,1054.94,True
RCC Model,6000,6000,0,1238.306550,0.17,1238.31,True


All recomputed values match the saved comparison.
All three averages match rounded thesis values: True
Historical status labels do not distinguish mathematical infeasibility from solver failure (ISSUE-09).


## Read legacy quadratic output (§4.5)

The experimental key is `(seed, dim, noise_std, P, alpha)`. Duplicate keys across
batches are rejected. Vector strings can be whitespace-separated NumPy output.
Missing/NaN `opt_true` is the available proxy for a true-infeasible instance, but
the old schema cannot distinguish solver failure. Legacy flags -1 and RCC [2]
indicate a missing learned solution; 0/1/False/True give true-model feasibility.

The exact historical aggregation rules are unavailable. Both feasibility
denominators are displayed. Timing filters are configurable. No aggregation can
remove RCC's construction/partitioning overhead from the recorded `t_rcc` values.


In [3]:
QUADRATIC_METHODS = {"abs": "Standard", "cqr": "CQR", "wrapper": "wrapper", "rcc": "RCC"}
QUADRATIC_KEY = ["seed", "dim", "noise_std", "P", "alpha"]

def parse_vector(value, dimension):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.lower() in {"", "nan", "none"}:
        return None
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1].strip()
    tokens = text.replace(",", " ").split()
    if not tokens:
        raise ValueError(f"Empty optimization vector: {value!r}")
    vector = np.array([float(token) for token in tokens])
    if np.isnan(vector).all():
        return None
    if len(vector) != dimension or not np.isfinite(vector).all():
        raise ValueError(f"Invalid vector/dimension {dimension}: {value!r}")
    return vector

def parse_flag(value):
    text = str(value).strip().lower()
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1].strip()
    if text in {"true", "false"}:
        return int(text == "true")
    number = float(text)
    if not np.isfinite(number) or number not in {-1, 0, 1, 2}:
        raise ValueError(f"Unknown feasibility flag: {value!r}")
    return int(number)

def load_quadratic(files, filters):
    required = set(QUADRATIC_KEY + ["opt_true"])
    for suffix in QUADRATIC_METHODS:
        required.update([f"opt_{suffix}", f"f_{suffix}", f"t_{suffix}"])
    frames = []
    for path in files:
        frame = read_result_csv(path)
        if not required.issubset(frame.columns):
            raise ValueError(f"{path.name}: missing columns {required - set(frame.columns)}")
        frame["source_file"] = str(path)
        frames.append(frame)
    if not frames:
        return None
    raw = pd.concat(frames, ignore_index=True)
    for key in QUADRATIC_KEY:
        raw[key] = pd.to_numeric(raw[key], errors="raise")
    if not np.isfinite(raw[QUADRATIC_KEY].to_numpy()).all():
        raise ValueError("Missing/nonfinite experimental key")
    for name in ["seed", "dim", "P"]:
        if not raw[name].eq(raw[name].astype(int)).all():
            raise ValueError(f"{name} must be integer-valued")
    if not raw["dim"].isin([1, 2, 3]).all() or (raw["P"] <= 0).any():
        raise ValueError("Unsupported dimension or invalid ensemble size")
    if raw.duplicated(QUADRATIC_KEY).any():
        raise ValueError("Duplicate experiment keys; select nonoverlapping batch CSVs")
    for key, allowed in filters.items():
        if key not in QUADRATIC_KEY:
            raise ValueError(f"Unknown filter: {key}")
        raw = raw.loc[raw[key].isin(allowed)].copy()
    raw["true_returned"] = [parse_vector(v, int(d)) is not None for v, d in zip(raw["opt_true"], raw["dim"])]
    for suffix in QUADRATIC_METHODS:
        raw[f"flag_{suffix}"] = raw[f"f_{suffix}"].map(parse_flag)
        raw[f"returned_{suffix}"] = [parse_vector(v, int(d)) is not None for v, d in zip(raw[f"opt_{suffix}"], raw["dim"])]
        if not raw[f"returned_{suffix}"].eq(raw[f"flag_{suffix}"].isin([0, 1])).all():
            raise ValueError(f"{suffix}: optimization vector and feasibility flag disagree")
        raw[f"t_{suffix}"] = pd.to_numeric(raw[f"t_{suffix}"], errors="raise")
        if not np.isfinite(raw[f"t_{suffix}"]).all() or (raw[f"t_{suffix}"] < 0).any():
            raise ValueError(f"{suffix}: invalid recorded time")
    return raw

def aggregate_quadratic(raw, feasibility_denominator, timing_population):
    if feasibility_denominator not in {"all_true_feasible", "returned_solutions"}:
        raise ValueError("Unknown feasibility denominator")
    if timing_population not in {"all_true_feasible", "returned_solutions", "all_instances"}:
        raise ValueError("Unknown timing population")
    feasible_rows = raw.loc[raw["true_returned"]]
    feasibility, timing = [], []
    for suffix, label in QUADRATIC_METHODS.items():
        flags = feasible_rows[f"flag_{suffix}"]
        n_instances, n_returned, n_feasible = len(flags), int(flags.isin([0, 1]).sum()), int(flags.eq(1).sum())
        all_rate = n_feasible / n_instances if n_instances else np.nan
        returned_rate = n_feasible / n_returned if n_returned else np.nan
        feasibility.append({"Method": label, "N_True_Feasible_Instances": n_instances,
                            "N_Returned": n_returned, "N_True_Feasible_Solutions": n_feasible,
                            "Rate_All_True_Feasible": all_rate, "Rate_Returned_Solutions": returned_rate,
                            "Selected_Rate": all_rate if feasibility_denominator == "all_true_feasible" else returned_rate})
        population = raw if timing_population == "all_instances" else feasible_rows
        if timing_population == "returned_solutions":
            population = population.loc[population[f"returned_{suffix}"]]
        for dimension in [1, 2, 3]:
            values = population.loc[population["dim"].eq(dimension), f"t_{suffix}"]
            timing.append({"dim": dimension, "Method": label, "N": len(values), "Mean_Recorded_Seconds": values.mean()})
    return pd.DataFrame(feasibility), pd.DataFrame(timing)

quadratic_raw = load_quadratic(QUADRATIC_FILES, QUADRATIC_FILTERS)
if quadratic_raw is None or quadratic_raw.empty:
    print("MISSING INPUTS: no selected quadratic observations; Tables 4.1–4.2 cannot be reproduced yet.")
    table_4_1 = pd.DataFrame(columns=["Method", "Selected_Rate"])
    table_4_2 = pd.DataFrame(columns=["dim", "Method", "N", "Mean_Recorded_Seconds"])
else:
    table_4_1, table_4_2 = aggregate_quadratic(quadratic_raw, FEASIBILITY_DENOMINATOR, TIMING_POPULATION)
    display(quadratic_raw.groupby(["dim", "noise_std", "P", "alpha"]).size().rename("N_Instances").reset_index())
    print("Selected rows:", len(quadratic_raw), "Excluded missing true solutions:", int((~quadratic_raw["true_returned"]).sum()))
    display(table_4_1)
    display(table_4_2.pivot(index="dim", columns="Method", values="Mean_Recorded_Seconds"))
    display(table_4_2.pivot(index="dim", columns="Method", values="N").rename_axis(columns="Timing sample counts"))
    print("ISSUE-08: recorded RCC time includes work excluded from other method timers.")


MISSING INPUTS: no selected quadratic observations; Tables 4.1–4.2 cannot be reproduced yet.


## Thesis reference values — not computed observations

These constants transcribe Tables 4.1–4.2 from the supplied thesis preview for
comparison only. A numerical match requires the historical inputs and inclusion
rules. Table 4.2 is a comparison of **recorded times** here, given ISSUE-08.


In [4]:
reference_4_1 = pd.DataFrame({"Method": ["Standard", "CQR", "wrapper", "RCC"],
                              "Thesis_Rate": [0.77, 0.92, 0.83, 0.95]})
reference_4_2 = pd.DataFrame({"dim": [1, 2, 3], "Standard": [0.01, 0.24, 1.80],
                              "CQR": [0.01, 0.07, 0.89], "wrapper": [0.05, 3.49, 61.25],
                              "RCC": [0.18, 3.92, 68.621]}).set_index("dim")
print("THESIS REFERENCES ONLY")
display(reference_4_1)
display(reference_4_2)
if not table_4_1.empty:
    comparison_4_1 = table_4_1.merge(reference_4_1, on="Method", validate="one_to_one")
    comparison_4_1["Difference"] = comparison_4_1["Selected_Rate"] - comparison_4_1["Thesis_Rate"]
    display(comparison_4_1[["Method", "Selected_Rate", "Thesis_Rate", "Difference"]])
    reference_long = reference_4_2.reset_index().melt(id_vars="dim", var_name="Method", value_name="Thesis_Seconds")
    comparison_4_2 = table_4_2.merge(reference_long, on=["dim", "Method"], validate="one_to_one")
    comparison_4_2["Difference_Seconds"] = comparison_4_2["Mean_Recorded_Seconds"] - comparison_4_2["Thesis_Seconds"]
    display(comparison_4_2)


THESIS REFERENCES ONLY


Method,Thesis_Rate
Standard,0.77
CQR,0.92
wrapper,0.83
RCC,0.95


Standard,CQR,wrapper,RCC
0.01,0.01,0.05,0.180
0.24,0.07,3.49,3.920
1.80,0.89,61.25,68.621


## Export tables and provenance

Exports contain computed observations only. Missing quadratic inputs yield empty
computed CSVs plus `missing_inputs` in the manifest, never the reference values.
The manifest records source hashes, filters, denominator choices, and table status.


In [5]:
aggregation_manifest = {
    "inputs_sha256": INPUT_HASHES,
    "quadratic_filters": QUADRATIC_FILTERS,
    "feasibility_denominator": FEASIBILITY_DENOMINATOR,
    "timing_population": TIMING_POPULATION,
    "table_4_3": {"status": "computed", "matches_rounded_thesis": bool(table_4_3["Matches_Thesis_2dp"].all())},
    "tables_4_1_4_2": {"status": "missing_inputs" if table_4_1.empty else "computed_with_unverified_historical_filters"},
    "limitations": ["No algorithm fixes", "Quadratic historical raw inputs/aggregator not recovered",
                    "RCC timing scope differs", "Missing true solution may also mean solver failure"],
}
if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    table_4_3.to_csv(EXPORT_DIR / "table_4_3.csv", index=False)
    table_4_1.to_csv(EXPORT_DIR / "table_4_1.csv", index=False)
    table_4_2.to_csv(EXPORT_DIR / "table_4_2_recorded_times.csv", index=False)
    (EXPORT_DIR / "aggregation_manifest.json").write_text(json.dumps(aggregation_manifest, indent=2) + "\n")
    print("Exported computed tables and provenance to", EXPORT_DIR.relative_to(ROOT))
print(json.dumps({"table_4_3": aggregation_manifest["table_4_3"],
                  "tables_4_1_4_2": aggregation_manifest["tables_4_1_4_2"]}, indent=2))


Exported computed tables and provenance to results/aggregated
{
  "table_4_3": {
    "status": "computed",
    "matches_rounded_thesis": true
  },
  "tables_4_1_4_2": {
    "status": "missing_inputs"
  }
}
